# Module 4: Component-level evals

1. **Exercise 5 (blame game):** three wrong answers. Which piece broke?
2. Retrieval metrics, and **Exercise 6:** label relevant chunks and measure retrieval.
3. Groundedness: does the answer stick to what was retrieved?
4. **Exercise 7:** checks on logged tool calls.

This afternoon the bot is **version 2**: an agent with tools. It can search the docs, look up orders, start
returns, check warranties, and hand off to a person with `create_ticket`. There's deliberately no refund tool.

In [ ]:
# Setup: run this cell first. It works in Google Colab and on your own laptop.
import os, sys
REPO_URL = "https://github.com/MarinaWyss/evaluating-ai-systems"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/EvalsWorkshop"):
        !git clone -q {REPO_URL} /content/EvalsWorkshop
        !pip install -q "litellm>=1.80.5" tenacity
    os.chdir("/content/EvalsWorkshop")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pandas as pd
from beefcake import llm
llm.load_colab_secrets()
if llm.has_api_key():
    print("API key found. Bot model:", llm.get_model())
else:
    print("No API key found, so this notebook runs in offline mode with pre-generated data.")

## 1. Exercise 5: Blame game (8 min)

Three wrong answers, with full traces. For each one, decide which piece broke: **retrieval** (it fetched the
wrong text), **prompt** (an instruction is missing), **generation** (it had the right text and still got it
wrong), or **tool call** (a tool was called wrong). First table with all three right wins.

In [ ]:
from beefcake.agents import load_agent_traces, show_agent_trace

blame = load_agent_traces("exercise5_blame_game.jsonl")
for t in blame:
    show_agent_trace(t, full=True)
    print()

In [ ]:
my_answers = {"B1": "", "B2": "", "B3": ""}  # retrieval, prompt, generation, or tool call

## 2. Retrieval metrics

| Metric | What it measures |
|---|---|
| Hit rate@k | Share of questions with a relevant chunk in the top k |
| Recall@k | Share of all relevant chunks that made the top k |
| Precision@k | Share of the top k that are relevant |
| MRR | Average of 1 / rank of the first relevant chunk |

### Exercise 6: Retrieval eval (12 min)

1. For each golden question, look at the top 5 chunks and write down the **section IDs** that actually answer it.
   Ask yourself: could the bot answer from this chunk alone? Mentioning the right product isn't enough.
2. Compute hit rate@3 and MRR.
3. Change one thing (`TOP_K`, or `MAX_WORDS` for the chunk size) and rerun.

In [ ]:
from beefcake.retrieval import BM25Retriever, load_chunks

golden = pd.read_csv("data/exercise6_questions.csv")["Question"].tolist()
candidates = BM25Retriever()
for q in golden:
    print("QUESTION:", q)
    for rank, chunk in enumerate(candidates.retrieve(q, k=5), 1):
        print(f"  {rank}. {chunk.section_id}\n     {chunk.text[:140]}...")
    print()

Every golden question has at least one relevant section somewhere in the docs. If none of the top 5 answer it,
find the right one in this list of all sections, so a miss counts as a miss.

In [ ]:
for c in load_chunks():
    print(f"{c.section_id:65} {c.text[:60]}...")

In [ ]:
relevant = {
    "how do i stop being charged after my free trial": [],
    "why did coach charge me twice": [],
    "rower wont turn on": [],
    "is rowing ok with a bad back": [],
    "can i return the kettlebell if i already used it": [],
    "how long is the warranty on the pulse": [],
    "bell dial is stuck between weights": [],
    "heart rate readings jump around when i start rowing": [],
    "where do i find the serial number on my bell": [],
    "how do i charge the kettlebell": [],
}

In [ ]:
from beefcake.retrieval_metrics import hit_rate, precision, recall, reciprocal_rank

TOP_K = 3          # try 5
MAX_WORDS = None   # try 35 or 20 to split long sections into smaller chunks

def retrieval_report(relevant, top_k=TOP_K, max_words=MAX_WORDS):
    retriever = BM25Retriever(load_chunks(max_words=max_words))
    rows = []
    for q, rel in relevant.items():
        ranked = []
        for c in retriever.retrieve(q, k=20):
            if c.section_id not in ranked:
                ranked.append(c.section_id)
        rows.append({"question": q, "retrieved": ranked[:top_k],
                     "hit": hit_rate(ranked, set(rel), top_k),
                     "RR": round(reciprocal_rank(ranked, set(rel), top_k), 2),
                     "precision": round(precision(ranked, set(rel), top_k), 2),
                     "recall": round(recall(ranked, set(rel), top_k), 2)})
    table = pd.DataFrame(rows)
    print(f"top_k={top_k}, max_words={max_words}:  hit rate {table['hit'].mean():.0%},  MRR {table['RR'].mean():.2f},"
          f"  precision {table['precision'].mean():.2f},  recall {table['recall'].mean():.2f}")
    return table

missing = [q for q, r in relevant.items() if not r]
if missing:
    print(f"Label every question first ({len(missing)} still empty). Leaving one out would hide a miss.")
else:
    display(retrieval_report(relevant))

Did MRR and hit rate move together when you changed something? If you tried a larger `TOP_K`, what happened to
precision, and to how long the prompt gets? Sections you never labeled count as not relevant, so if a new
setting surfaces one, label it.

## 3. Groundedness

Split the answer into claims and check each claim against the retrieved text. One unsupported claim is enough to
flag the answer. Here's round 2 of the blame game, checked by an LLM. Like any judge, this checker needs
validating against human labels before you rely on it.

In [ ]:
from beefcake.judge import groundedness
from beefcake.tools import run_tool

b2 = blame[1]
context = "\n\n".join(run_tool("search_docs", {"query": "rower warranty"})["results"])
print("ANSWER:", b2.final_answer, "\n")
if llm.has_api_key():
    display(groundedness(b2.final_answer, context))
else:
    print("Needs an API key. By hand: the '3-year warranty' claim isn't supported, because the policy says 2 years.")

## 4. Exercise 7: Tool-call eval (8 min)

Here are logged tool calls from version 2, and what we expected for each. Finish the four checks, run them,
and see which traces fail. Things you can use:

- `first_action_tool(trace)` gives the first tool that isn't a doc search (or `"none"`)
- `schema_errors(name, arguments)` lists problems with a call's arguments
- `APPROVED_TOOLS` is the set of real tool names
- `trace.tool_calls` is the list of tool-call spans (each has `name` and `input`), and `trace.end_state` is what
  the store looks like afterwards

In [ ]:
import json
from beefcake.checks import first_action_tool
from beefcake.tools import APPROVED_TOOLS, schema_errors

tool_traces = {t.trace_id: t for t in load_agent_traces("exercise7_tool_calls.jsonl")}
expected = pd.read_csv("data/exercise7_expected.csv").set_index("Trace ID")
show_agent_trace(tool_traces["C04"])
expected

In [ ]:
def check_expected_tool(trace, exp):
    # YOUR CODE HERE: True if the first action tool matches exp["Expected first tool"]
    return True

def check_schema(trace, exp):
    # YOUR CODE HERE: True if every approved tool call has valid arguments
    return True

def check_no_made_up_tools(trace, exp):
    # YOUR CODE HERE: True if every tool call is in APPROVED_TOOLS
    return True

def check_end_state(trace, exp):
    # YOUR CODE HERE: compare trace.end_state with json.loads(exp["Expected returns"]) and exp["Expected tickets"]
    return True

CHECKS = {"expected tool": check_expected_tool, "schema": check_schema,
          "no made-up tools": check_no_made_up_tools, "end state": check_end_state}

def run_tool_checks(traces, checks_to_run):
    return pd.DataFrame([
        {"Trace ID": tid, **{name: "PASS" if fn(t, expected.loc[tid]) else "FAIL" for name, fn in checks_to_run.items()}}
        for tid, t in traces.items()
    ])

run_tool_checks(tool_traces, CHECKS)

One of these traces takes a different path to the right end state. Did any of your checks fail it? Should it
have failed? Also look for traces that pass every argument check and still leave the store in the wrong state.

### Stretch: run version 2 live

With a key, run the same questions through the live agent and apply your checks. Your model will make
different mistakes from the logged ones.

In [ ]:
from beefcake.agents import answer_v2

if llm.has_api_key():
    live = {tid: answer_v2(expected.loc[tid, "User Query"], trace_id=tid) for tid in expected.index[:5]}
    display(run_tool_checks(live, CHECKS))
    show_agent_trace(live["C01"])
else:
    print("Needs an API key.")

---
## Solutions (try it yourself first)

In [ ]:
from beefcake import checks

SOLUTION_CHECKS = {
    "expected tool": lambda t, e: checks.first_action_tool(t) == e["Expected first tool"],
    "schema": lambda t, e: checks.arguments_match_schema(t),
    "no made-up tools": lambda t, e: checks.only_approved_tools(t),
    "end state": lambda t, e: checks.end_state_matches(t, json.loads(e["Expected returns"]), int(e["Expected tickets"])),
    "no false success": lambda t, e: checks.no_false_success(t),
}
run_tool_checks(tool_traces, SOLUTION_CHECKS)